# Dynamic Memory

---

## Memory Layout

- Stack - for local variables, function arguments and return addresses
- Heap - for dynamically allocated memory
- Global/Static - global and static variables
- Code - the program instructions

<center>

![memory layout](img/memory-layout.svg)

</center>

---

## Stack

Function arguments, and all variables that are local to that function, are stored on the stack. 

So, whenever a function is called:
- ⤵ Push arguments
- ⤵ Push return address
- ↷ Jump to memory location of the function code

When the function is finished executing:
- ⤴ Pop local variables and arguments off the stack
- ⤵ Push the return value onto the stack
- ↷ Jump back to the return address

---

## Heap

The heap manages memory dynamically allocated at run-time. While the stack stores memory associated with called functions, the heap is associated with memory that needs to be managed by the programmer.

When we request an allocation of memory from the heap, if there is sufficient contiguous memory available, we are given the address to the start of the allocated memory.

---

## Memory Allocation

Memory allocation functions (part of `<stdlib.h>`) return void pointers `void*`, which are pointers to memory where the type isn't necessarily known. So when we allocate memory, the pointer will need to be cast to a specific type.

### malloc

Requests `size` bytes required. Each data type has a different size, so `sizeof` will be useful for this. (Can also allocate 0 bytes of memory).

Returns a pointer to the allocated memory if successful, or `NULL` if unsuccessful.
```c
void* malloc(size_t size);
```

### calloc

Requests `num` blocks of memory, each `size` bytes, and set to `0`.
```c
void* calloc(size_t num, size_t size);
```

### realloc

Attempts to resize previously allocated memory. This may require a new block of memory to be found, so it returns a new void pointer to memory.

If unsuccessful, the current memory is preserved, and `NULL` is returned.
```c
void* realloc(void* ptr, size_t size);
```

### free

De-allocates previously allocated memory.
```c
void free(void* ptr);
```

---

## Security

1. It is good practice to de-allocate memory that is no longer required.

2. It is also a common error to try and free memory that has already been de-allocated, or that has not been allocated using heap memory.

3. Accessing memory that has been de-allocated can lead to serious problems.

4. Be cautious with size; remember that strings include a null terminator `'\0'`.

5. You can never assume that memory has been successfully allocated; check for when `malloc`/`calloc`/`realloc` return `NULL`.

---

## Dynamic Arrays

For arrays that depend on run-time behaviour, create them on the heap.
```c
struct dyn_arr {
    int* arr;
    size_t size;
    size_t length;
};

struct dyn_arr* my_arr = (struct dyn_arr *) malloc(sizeof(struct dyn_arr));

my_arr->size = 2;
my_arr->length = 0;

my_arr->arr = (int *) calloc(my_arr->size, sizeof(int));
```
A common practice is to double the size of the array every time it becomes full (this way, appending $n$ elements takes $O(n)$ time).
```c
void append(struct dyn_arr* my_arr, int a) {
    my_arr->arr[my_arr->length] = a;
    my_arr->length++;
    
    if (my_arr->length == my_arr->size) {
        my_arr->size *= 2;
        my_arr->arr = (int *) realloc(my_arr->arr, my_arr->size * sizeof(int));
    }
}
```

---
---

# Linked Lists

---

## Internals

Linked lists in C follow a dynamic structure. We must explicitly allocate and free each node within the list.
```c
struct node {
    // declare data here...
    struct node* next;
};
struct l_list {
    struct node* head;
};
```
Each element in our singly-linked list has two components:
- The data (can be anything you want)
- The link, a pointer to the next node (or `NULL` if it is last)

---

## Characteristics

Linked lists have $O(n)$ cost of access, compared to arrays which are $O(1)$.

An advantage to linked lists, however, is their flexible structure. Inserts and removes only require the links to be changed, as opposed to dynamic arrays where you would need to copy over large chunks of memory.

---

## Insert

Inserting into the start of a linked list is simple; prepend to the start of the list, and update the head.
```c
void prepend(struct l_list* list, struct node* elem) {
    elem->next = list->head;
    list->head = elem;
}
```
For inserting at a specified position, we can traverse the list to find the node before and reorder its links.
```c
int l_insert(struct l_list* list, struct node* elem, int pos) {
    if (pos == 0) {
        prepend(list, elem);
        return 0;
    }

    struct node* current = list->head;
    for (int i = 1; current != NULL && i < pos; i++) {
        current = current->next;
    }
    if (current == NULL) return -1; // pos out of bounds

    elem->next = current->next;
    current->next = elem;
    return 0;
}
```

---

## Remove

To remove the head we can pop the top of the list (remember to `free` it afterwards).
```c
struct node* pop(struct l_list* list) {
    struct node* elem = list->head;
    if (elem == NULL) return NULL;

    list->head = elem->next;
    elem->next = NULL;
    return elem;
}
```

Now traverse the list until we reach the node we want to delete.
```c
int l_remove(struct l_list* list, int pos) {
    if (pos == 0) {
        struct node* elem = pop(list);
        if (elem == NULL) return -1;
        free(elem);
        return 0;
    }

    struct node* current = list->head;
    for (int i = 1; current != NULL && i < pos; i++) {
        current = current->next;
    }
    if (current == NULL) return -1; // pos out of bounds

    struct node* elem = current->next;
    if (elem == NULL) return -1;
    current->next = elem->next;
    free(elem);
    return 0;
}
```

---

## Extras

### Doubly Linked Lists

In addition to having a pointer to the `next` node, you also have a pointer to the `prev` node. This generally just means that instead of having to perform 2 link changes with inserts/removes, you now do 4.

### Circular Linked Lists

A slight variation on the standard linked list, where the last node points to the head rather than `NULL`. 

It is also possible to have a circular doubly linked list, where you can have simpler access to the previous node. And, thanks to the circular structure, there are less edge cases concerning the start/end of the list.